# 01 — DQN with Pixel Observations
Train a DQN agent on Super Mario Bros using pixel-based input (grayscale, 84x84, 4-frame stack).

In [ ]:
# --- Google Colab Setup ---
import os, sys

if os.getenv('COLAB_RELEASE_TAG'):
    !pip install -q Cython setuptools wheel
    !git clone -b hotfix/version1 https://github.com/lmartim4/CSC-52081-ContinousMountainCar.git /content/repo
    %cd /content/repo
    !pip install -r requirements.txt --no-build-isolation
    sys.path.insert(0, '/content/repo')
    import site; SITE = site.getsitepackages()[0]
    !patch --forward -p0 {SITE}/nes_py/_rom.py                   < patches/nes_py_numpy2.patch || true
    !patch --forward -p0 {SITE}/gym/utils/passive_env_checker.py < patches/gym_bool8_numpy2.patch || true
    !patch --forward -p0 {SITE}/gym_super_mario_bros/smb_env.py  < patches/smb_env_numpy2.patch || true
    !sed -i 's/observation, reward, terminated, truncated, info = self.env.step(action)/_result = self.env.step(action); observation, reward, terminated, info = _result[:4]; truncated = _result[4] if len(_result) > 4 else False/' {SITE}/gym/wrappers/time_limit.py
else:
    if os.path.basename(os.getcwd()) == 'notebooks':
        %cd ..

In [ ]:
from src.wrappers import make_pixel_env
from src.agents import make_dqn_agent
from src.utils.callbacks import CheckpointAndLogCallback
from src.config import DQNConfig

In [ ]:
# Create environment
env = make_pixel_env()
print(f'Observation space: {env.observation_space.shape}')
print(f'Action space: {env.action_space.n}')

In [ ]:
# Configure and create agent
config = DQNConfig(total_timesteps=2_000_000, policy='CnnPolicy')
model = make_dqn_agent(env, config=config, tensorboard_log='../logs/pixel_dqn')

In [ ]:
# Train
callback = CheckpointAndLogCallback(
    save_path='../models/pixel_dqn',
    save_freq=100_000,
)
model.learn(total_timesteps=config.total_timesteps, callback=callback)
model.save('../models/pixel_dqn/final_model')

In [ ]:
# Quick test
from src.utils.evaluation import evaluate_agent
results = evaluate_agent(model, env, n_episodes=10)
print(f"Mean reward: {results['mean_reward']:.1f} ± {results['std_reward']:.1f}")
print(f"Flag rate: {results['flag_rate']:.2%}")